In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error

In [2]:
Y_COL = 'Sum of кВт'

In [9]:
def smallest_int_dtype(min_val: int, max_val: int, signed: bool = True) -> str:
    if signed:
        if np.iinfo(np.int8).min <= min_val <= max_val <= np.iinfo(np.int8).max:
            return "int8"
        if np.iinfo(np.int16).min <= min_val <= max_val <= np.iinfo(np.int16).max:
            return "int16"
        if np.iinfo(np.int32).min <= min_val <= max_val <= np.iinfo(np.int32).max:
            return "int32"
        return "int64"
    else:
        if 0 <= min_val <= max_val <= np.iinfo(np.uint8).max:
            return "uint8"
        if 0 <= min_val <= max_val <= np.iinfo(np.uint16).max:
            return "uint16"
        if 0 <= min_val <= max_val <= np.iinfo(np.uint32).max:
            return "uint32"
        return "uint64"


def optimize_df_for_memory(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Converts columns to smaller dtypes.
    For low-decimal float columns, stores scaled integers if that beats float32.
    Returns:
        optimized_df
        metadata dict with scaling info
    """
    meta = {}

    for col in df.columns:
        s = df[col]

        # bool-like columns
        unique_non_null = set(s.dropna().unique())
        if unique_non_null.issubset({0, 1, True, False}):
            if col == "Група":
                df[col] = s.astype("bool")
                meta[col] = {"stored_as": "bool", "scale": 1}
                continue

        # integer columns
        if pd.api.types.is_integer_dtype(s):
            mn, mx = int(s.min()), int(s.max())
            dtype = smallest_int_dtype(mn, mx, signed=(mn < 0))
            df[col] = s.astype(dtype)
            meta[col] = {"stored_as": dtype, "scale": 1}
            continue

        # float columns
        if pd.api.types.is_float_dtype(s):
            # estimate visible decimal precision
            non_null = s.dropna()
            if len(non_null) == 0:
                df[col] = s.astype("float32")
                meta[col] = {"stored_as": "float32", "scale": 1}
                continue

            decimals = non_null.astype(str).apply(
                lambda x: len(x.split(".")[1].rstrip("0")) if "." in x else 0
            ).max()

            # try scaled integer
            if decimals <= 3:
                scale = 10 ** decimals
                scaled = np.round(s * scale)

                mn = int(np.nanmin(scaled))
                mx = int(np.nanmax(scaled))
                int_dtype = smallest_int_dtype(mn, mx, signed=(mn < 0))

                int_bytes = np.dtype(int_dtype).itemsize
                float32_bytes = np.dtype("float32").itemsize

                if int_bytes < float32_bytes:
                    df[col] = scaled.astype(int_dtype)
                    meta[col] = {"stored_as": int_dtype, "scale": scale}
                else:
                    df[col] = s.astype("float32")
                    meta[col] = {"stored_as": "float32", "scale": 1}
            else:
                df[col] = s.astype("float32")
                meta[col] = {"stored_as": "float32", "scale": 1}

    return df, meta

def add_cat_helpers(df: pd.DataFrame) -> pd.DataFrame:
    """Create string categoricals from datetime for PyTorch Forecasting."""

    # ensure datetime column is in datetime format
    df['datetime'] = pd.to_datetime(df['datetime'])

    # basic time components
    # df['Year_cat'] = df['datetime'].dt.year
    df['Month_cat'] = df['datetime'].dt.month.astype(str)
    df['Day_cat'] = df['datetime'].dt.day.astype(str)
    df['Hour_cat'] = df['datetime'].dt.hour.astype(str)

    # day of week (0=Monday)
    df['day_of_week_cat'] = df['datetime'].dt.dayofweek.astype(str)

    # season mapping
    def get_season(month):
        if month in [12, 1, 2]:
            return 'winter'
        elif month in [3, 4, 5]:
            return 'spring'
        elif month in [6, 7, 8]:
            return 'summer'
        else:
            return 'autumn'

    df['season_cat'] = df['datetime'].dt.month.map(get_season)

    return df

def load_and_prepare(path: str) -> pd.DataFrame:
    df = pd.read_parquet(path).reset_index(drop=True)

    df.columns = df.columns.str.replace('.', '_', regex=False)
    df, _ = optimize_df_for_memory(df)
    df = add_cat_helpers(df)
    for col in df.columns:
        if col.endswith("_cat"):
            df[col] = df[col].astype(str)
    try:
        df[Y_COL] = df[Y_COL].astype("float32")
    except:
        print(f"No Y_col: {Y_COL}")
    df = df.sort_values(['EIC-код_cat', "datetime"]).reset_index(drop=True)
    try:
        df.drop(columns=["Ціна розподілу ЕЕ", "Ціна ЕЕ", "Money_spent"], inplace=True)
    except:
        print("No price columns")

    return df

In [4]:
train = load_and_prepare("data/silver_money_calc/train.parquet")

val = load_and_prepare("data/silver_money_calc/val.parquet")

,EIC-код_cat,Група_cat,Sum of кВт,АЗС_cat,Тип_cat,Область_cat,Широта,Довгота,ОСР код_cat,ОСР опис_cat,...,wind_direction_10m,wind_gusts_10m,shortwave_radiation,diffuse_radiation,direct_normal_irradiance,Month_cat,Day_cat,Hour_cat,day_of_week_cat,season_cat
0,62Z0008583037334,0,24.043646,АЗС_29,ОККО-комплекс,Закарпатська,48.442429,22.192190,MGA-00700,Ужгород,...,125,216,0,0,0,1,1,1,0,winter
1,62Z0008583037334,0,22.768120,АЗС_29,ОККО-комплекс,Закарпатська,48.442429,22.192190,MGA-00700,Ужгород,...,131,176,0,0,0,1,1,2,0,winter
2,62Z0008583037334,0,21.620148,АЗС_29,ОККО-комплекс,Закарпатська,48.442429,22.192190,MGA-00700,Ужгород,...,122,187,0,0,0,1,1,3,0,winter
3,62Z0008583037334,0,20.791056,АЗС_29,ОККО-комплекс,Закарпатська,48.442429,22.192190,MGA-00700,Ужгород,...,135,223,0,0,0,1,1,4,0,winter
4,62Z0008583037334,0,20.344624,АЗС_29,ОККО-комплекс,Закарпатська,48.442429,22.192190,MGA-00700,Ужгород,...,143,216,0,0,0,1,1,5,0,winter
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4864319,62Z9997819406173,0,22.032907,АЗС_65,ОККО-міська,Львівська,49.800007,24.068964,MGA-00900,Львів,...,322,436,215,75,4769,6,30,19,0,summer
4864320,62Z9997819406173,0,22.187634,АЗС_65,ОККО-міська,Львівська,49.800007,24.068964,MGA-00900,Львів,...,342,335,87,41,3204,6,30,20,0,summer
4864321,62Z9997819406173,0,22.404249,АЗС_65,ОККО-міська,Львівська,49.800007,24.068964,MGA-00900,Львів,...,331,302,6,5,0,6,30,21,0,summer
4864322,62Z9997819406173,0,23.085041,АЗС_65,ОККО-міська,Львівська,49.800007,24.068964,MGA-00900,Львів,...,318,266,0,0,0,6,30,22,0,summer


In [ ]:
def smape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return np.mean(np.abs(y_true - y_pred) / np.maximum(denom, 1e-8))

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

In [ ]:
# model training script here

In [ ]:
test = load_and_prepare("data/silver_money_calc/test.parquet")

In [ ]:
# inference script here

In [ ]:
print("\nValidation metrics")
print("SMAPE:", smape(val[Y_COL], val_pred))
print("RMSE :", rmse(val[Y_COL], val_pred))
print("MAPE :", mape(val[Y_COL], val_pred))

print("\nTest metrics")
print("SMAPE:", smape(test[Y_COL], test_pred))
print("RMSE :", rmse(test[Y_COL], test_pred))
print("MAPE :", mape(test[Y_COL], test_pred))

In [10]:
predict_X = load_and_prepare("data/no_y_col/with_weather_v2.parquet")
predict_X

No Y_col: Sum of кВт
No price columns


,EIC-код_cat,datetime,Група_cat,АЗС_cat,Тип_cat,Область_cat,Широта,Довгота,ОСР код_cat,ОСР опис_cat,...,wind_direction_10m,wind_gusts_10m,shortwave_radiation,diffuse_radiation,direct_normal_irradiance,Month_cat,Day_cat,Hour_cat,day_of_week_cat,season_cat
0,62Z0008583037334,2025-09-01 00:00:00+03:00,1,АЗС_29,ОККО-комплекс,Закарпатська,48.442429,22.192190,MGA-00700,Ужгород,...,337,299,0,0,0,9,1,0,0,autumn
1,62Z0008583037334,2025-09-01 01:00:00+03:00,1,АЗС_29,ОККО-комплекс,Закарпатська,48.442429,22.192190,MGA-00700,Ужгород,...,333,295,0,0,0,9,1,1,0,autumn
2,62Z0008583037334,2025-09-01 02:00:00+03:00,1,АЗС_29,ОККО-комплекс,Закарпатська,48.442429,22.192190,MGA-00700,Ужгород,...,333,274,0,0,0,9,1,2,0,autumn
3,62Z0008583037334,2025-09-01 03:00:00+03:00,1,АЗС_29,ОККО-комплекс,Закарпатська,48.442429,22.192190,MGA-00700,Ужгород,...,332,277,0,0,0,9,1,3,0,autumn
4,62Z0008583037334,2025-09-01 04:00:00+03:00,1,АЗС_29,ОККО-комплекс,Закарпатська,48.442429,22.192190,MGA-00700,Ужгород,...,335,252,0,0,0,9,1,4,0,autumn
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025417,62Z9997819406173,2026-03-31 20:00:00+03:00,0,АЗС_65,ОККО-міська,Львівська,49.800007,24.068964,MGA-00900,Львів,...,340,428,3,3,0,3,31,20,1,spring
2025418,62Z9997819406173,2026-03-31 21:00:00+03:00,0,АЗС_65,ОККО-міська,Львівська,49.800007,24.068964,MGA-00900,Львів,...,341,421,0,0,0,3,31,21,1,spring
2025419,62Z9997819406173,2026-03-31 22:00:00+03:00,0,АЗС_65,ОККО-міська,Львівська,49.800007,24.068964,MGA-00900,Львів,...,343,407,0,0,0,3,31,22,1,spring
2025420,62Z9997819406173,2026-03-31 23:00:00+03:00,0,АЗС_65,ОККО-міська,Львівська,49.800007,24.068964,MGA-00900,Львів,...,343,421,0,0,0,3,31,23,1,spring


In [ ]:
# prediction script for predict_X

In [11]:
train.columns.tolist()

['EIC-код_cat',
 'Група_cat',
 'Sum of кВт',
 'АЗС_cat',
 'Тип_cat',
 'Область_cat',
 'Широта',
 'Довгота',
 'ОСР код_cat',
 'ОСР опис_cat',
 'datetime',
 'temperature_2m',
 'apparent_temperature',
 'dew_point_2m',
 'relative_humidity_2m',
 'precipitation',
 'rain',
 'snowfall',
 'cloud_cover',
 'cloud_cover_low',
 'cloud_cover_mid',
 'cloud_cover_high',
 'surface_pressure',
 'wind_speed_10m',
 'wind_direction_10m',
 'wind_gusts_10m',
 'shortwave_radiation',
 'diffuse_radiation',
 'direct_normal_irradiance',
 'Month_cat',
 'Day_cat',
 'Hour_cat',
 'day_of_week_cat',
 'season_cat']

In [12]:
predict_X.columns.tolist()

['EIC-код_cat',
 'datetime',
 'Група_cat',
 'АЗС_cat',
 'Тип_cat',
 'Область_cat',
 'Широта',
 'Довгота',
 'ОСР код_cat',
 'ОСР опис_cat',
 'temperature_2m',
 'apparent_temperature',
 'dew_point_2m',
 'relative_humidity_2m',
 'precipitation',
 'rain',
 'snowfall',
 'cloud_cover',
 'cloud_cover_low',
 'cloud_cover_mid',
 'cloud_cover_high',
 'surface_pressure',
 'wind_speed_10m',
 'wind_direction_10m',
 'wind_gusts_10m',
 'shortwave_radiation',
 'diffuse_radiation',
 'direct_normal_irradiance',
 'Month_cat',
 'Day_cat',
 'Hour_cat',
 'day_of_week_cat',
 'season_cat']